In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_data_validation as tfdv
from tensorflow_metadata.proto.v0 import schema_pb2
print('TF: {}'.format(tf.__version__))
print('TFDV version:', tfdv.version.__version__)

#pd.set_option("display.max_rows", None)



TF: 2.17.0
TFDV version: 1.14.0


In [18]:
CSV_DATA = 'dataset/laptop_data_1M_v2.csv'

In [19]:
df = pd.read_csv(CSV_DATA)
df.head()

,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price
0,Lenovo,2 in 1 Convertible,10.1,IPS Panel Touchscreen 1920x1200,Intel Atom x5-Z8550 1.44GHz,4GB,64GB Flash Storage,Intel HD Graphics 400,Windows 10,0.69kg,25521.12
1,Lenovo,Notebook,15.6,IPS Panel Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,1TB HDD,NaN,Windows 10,2.3kg,45323.1648
2,HP,Notebook,15.6,Full HD 1920x1080,AMD A9-Series 9410 2.9GHz,6GB,1.0TB Hybrid,AMD Radeon R7 M440,Windows 10,2.04kg,29303.4672
3,HP,Notebook,15.6,1366x768,Intel Core i7 7500U 2.7GHz,8GB,2TB HDD,Intel HD Graphics 620,Windows 10,2.04kg,33513.12
4,Acer,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD + 1TB HDD,Nvidia GeForce GTX 950M,Windows 10,2.4kg,42570.72


In [20]:
# Get all string columns (object dtype or explicitly known)
string_cols = df.select_dtypes(include='object').columns
# Fill NaN with "unknown"
df[string_cols] = df[string_cols].fillna('unknown')

In [21]:
for str_col in string_cols:
    print()
    print(df[str_col].value_counts(dropna=False))


Lenovo       222555
Dell         219929
HP           203967
Asus         119924
Acer          79243
MSI           40837
Toshiba       35880
unknown       23194
Apple         16259
Samsung        6851
Razer          5374
Mediacom       5337
Microsoft      4605
Xiaomi         3082
Vero           3025
Google         2297
Chuwi          2258
LG             2239
Fujitsu        1583
Huawei         1561
Name: Company, dtype: int64

Notebook              544521
Gaming                156131
Ultrabook             146670
2 in 1 Convertible     88812
unknown                23194
Workstation            22157
Netbook                18515
Name: TypeName, dtype: int64

15.6       490983
14         147137
17.3       124550
13.3       123481
12.5        29399
11.6        23595
unknown     23194
12           4649
13.9         4589
13.5         3869
12.3         3809
15           3088
15.4         3049
35.6         2339
10.1         2237
13           1561
24           1524
18.4          830
?            

In [22]:
print(df['Price'].value_counts(dropna=False))

unknown        22744
58554.72       10559
79866.72       10479
95850.72       10402
not_a_price    10000
               ...  
78647.1408       694
36443.52         687
93181.392        683
163723.5792      681
29144.16         654
Name: Price, Length: 781, dtype: int64


In [23]:
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
median = df['Price'].median()
df['Price'] = df['Price'].fillna(median)
print(df['Price'].value_counts(dropna=False))

52054.5600     35039
58554.7200     10559
79866.7200     10479
95850.7200     10402
63882.7200      8335
               ...  
78647.1408       694
36443.5200       687
93181.3920       683
163723.5792      681
29144.1600       654
Name: Price, Length: 779, dtype: int64


In [24]:
df['Ram'].value_counts(dropna=False)

8GB        452356
4GB        275615
16GB       145703
unknown     32731
6GB         29961
12GB        18849
2GB         16561
32GB        12910
1234TB      10000
64GB         2283
24GB         2272
1GB           759
Name: Ram, dtype: int64

In [25]:
df['Ram'] = df['Ram'].str.replace('GB', '', regex=False)
df['Ram'] = pd.to_numeric(df['Ram'], errors="coerce")
mean = df['Ram'].mean()
df['Ram'] = df['Ram'].fillna(mean)
df['Ram'].value_counts(dropna=False)

8.000000     452356
4.000000     275615
16.000000    145703
8.467997      42731
6.000000      29961
12.000000     18849
2.000000      16561
32.000000     12910
64.000000      2283
24.000000      2272
1.000000        759
Name: Ram, dtype: int64

In [26]:
df['Inches'].value_counts(dropna=False)

15.6       490983
14         147137
17.3       124550
13.3       123481
12.5        29399
11.6        23595
unknown     23194
12           4649
13.9         4589
13.5         3869
12.3         3809
15           3088
15.4         3049
35.6         2339
10.1         2237
13           1561
24           1524
18.4          830
?             782
27.3          781
25.6          779
14.1          773
17            769
31.6          749
11.3          748
33.5          736
Name: Inches, dtype: int64

In [27]:
df['Inches'] = pd.to_numeric(df['Inches'], errors='coerce')
mode = df['Inches'].mode()[0]
df['Inches'] = df['Inches'].fillna(mode)
print(df['Inches'].value_counts(dropna=False))

15.6    514959
14.0    147137
17.3    124550
13.3    123481
12.5     29399
11.6     23595
12.0      4649
13.9      4589
13.5      3869
12.3      3809
15.0      3088
15.4      3049
35.6      2339
10.1      2237
13.0      1561
24.0      1524
18.4       830
27.3       781
25.6       779
14.1       773
17.0       769
31.6       749
11.3       748
33.5       736
Name: Inches, dtype: int64


In [28]:
df.select_dtypes(include='number').columns

Index(['Inches', 'Ram', 'Price'], dtype='object')

In [29]:
df['Weight'].value_counts(dropna=False)

2.2kg     85177
2.1kg     43718
2.4kg     33088
2.3kg     31581
2.5kg     28328
          ...  
3.74kg      732
3.6kg       728
2.79kg      727
3.52kg      723
1.70kg      721
Name: Weight, Length: 190, dtype: int64

In [30]:
df['Weight'] = df['Weight'].str.replace('kg', '')
df['Weight'] = pd.to_numeric(df['Weight'], errors="coerce")
mode = df['Weight'].mode()[0]
df['Weight'] = df['Weight'].fillna(mode)
df['Weight'].value_counts(dropna=False)

2.20    112965
2.10     43718
2.40     33088
2.00     32294
2.30     31581
         ...  
2.21       739
3.74       732
3.60       728
2.79       727
3.52       723
Name: Weight, Length: 180, dtype: int64

In [32]:
df.select_dtypes(include='number').columns

Index(['Inches', 'Ram', 'Weight', 'Price'], dtype='object')

In [33]:
df.to_csv("dataset/laptop_data_1M_v2_CLEANED.csv", index=False)

In [34]:
cdf = pd.read_csv("dataset/laptop_data_1M_v2_CLEANED.csv")

for column in cdf.columns:
    print(f"========== VALUE COUNTS: [{column}] =================")
    print(cdf[column].value_counts(dropna=False))
    print()
    print(f"NULL values: {cdf[column].isnull().sum()}")

========== VALUE COUNTS: [Company] =================
Lenovo       222555
Dell         219929
HP           203967
Asus         119924
Acer          79243
MSI           40837
Toshiba       35880
unknown       23194
Apple         16259
Samsung        6851
Razer          5374
Mediacom       5337
Microsoft      4605
Xiaomi         3082
Vero           3025
Google         2297
Chuwi          2258
LG             2239
Fujitsu        1583
Huawei         1561
Name: Company, dtype: int64

NULL values: 0
========== VALUE COUNTS: [TypeName] =================
Notebook              544521
Gaming                156131
Ultrabook             146670
2 in 1 Convertible     88812
unknown                23194
Workstation            22157
Netbook                18515
Name: TypeName, dtype: int64

NULL values: 0
========== VALUE COUNTS: [Inches] =================
15.6    514959
14.0    147137
17.3    124550
13.3    123481
12.5     29399
11.6     23595
12.0      4649
13.9      4589
13.5      3869
12.3      3809